In [10]:
import gc
import pickle

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Baseline (raw features only)

## Load artifacts

In [11]:
DATA_DIR = '/kaggle/input/datasets/f0rzah/artifacts3/'
RAW_DIR = '/kaggle/input/datasets/f0rzah/kion-dataset/data_original/'

train_final = pd.read_parquet(f'{DATA_DIR}train_final.parquet')
items_df = pd.read_csv(f'{RAW_DIR}items.csv')
users_df = pd.read_csv(f'{RAW_DIR}users.csv')

with open(f'{DATA_DIR}evaluation_samples.pkl', 'rb') as f:
    eval_data = pickle.load(f)

print(f'train_final: {train_final.shape}')

train_final: (9178632, 25)


## Raw feature tables

In [12]:
items_raw = items_df[['item_id', 'content_type', 'release_year', 'age_rating', 'for_kids']].copy()
items_raw['content_type'] = items_raw['content_type'].astype(str)
items_raw['release_year'] = items_raw['release_year'].astype('float32')
items_raw['age_rating'] = items_raw['age_rating'].astype('float32')
items_raw['for_kids'] = items_raw['for_kids'].fillna(0).astype('int8')

users_raw = users_df[['user_id', 'age', 'sex', 'income', 'kids_flg']].copy()
users_raw['age'] = users_raw['age'].fillna('Unknown').astype(str)
users_raw['sex'] = users_raw['sex'].fillna('Unknown').astype(str)
users_raw['income'] = users_raw['income'].fillna('Unknown').astype(str)
users_raw['kids_flg'] = users_raw['kids_flg'].fillna(0).astype('int8')

print(f'items_raw: {items_raw.shape}, users_raw: {users_raw.shape}')

items_raw: (15963, 5), users_raw: (840197, 5)


## Assemble training data

In [13]:
base = train_final[['user_id', 'item_id', 'target']].copy()
base = base.merge(users_raw, on='user_id', how='left')
base = base.merge(items_raw, on='item_id', how='left')
CAT_FEATURES = ['content_type', 'age', 'sex', 'income']
for col in CAT_FEATURES:
    base[col] = base[col].astype(str)
print(f'base: {base.shape}')

base: (9178632, 11)


## Feature schema

In [14]:
CAT_FEATURES = ['content_type', 'age', 'sex', 'income']
NUM_FEATURES = ['kids_flg', 'release_year', 'age_rating', 'for_kids']
FEATURES = CAT_FEATURES + NUM_FEATURES

print(f'features ({len(FEATURES)}): {FEATURES}')

features (8): ['content_type', 'age', 'sex', 'income', 'kids_flg', 'release_year', 'age_rating', 'for_kids']


## Train / internal val split

In [15]:
X = base[FEATURES]
y = base['target']

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

del X, y
gc.collect()

19

## Train CatBoost

In [16]:
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='Logloss',
    eval_metric='Logloss',
    random_seed=42,
    verbose=100,
    cat_features=CAT_FEATURES,
    task_type='GPU',
    devices='0',
    early_stopping_rounds=50,
    use_best_model=True,
)

model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

0:	learn: 0.6719475	test: 0.6719777	best: 0.6719777 (0)	total: 697ms	remaining: 5m 47s
100:	learn: 0.4893080	test: 0.4898787	best: 0.4898787 (100)	total: 56.6s	remaining: 3m 43s
200:	learn: 0.4850217	test: 0.4855965	best: 0.4855965 (200)	total: 1m 48s	remaining: 2m 41s
300:	learn: 0.4829594	test: 0.4835505	best: 0.4835505 (300)	total: 2m 40s	remaining: 1m 46s
400:	learn: 0.4812566	test: 0.4818451	best: 0.4818451 (400)	total: 3m 32s	remaining: 52.5s
499:	learn: 0.4799388	test: 0.4805097	best: 0.4805097 (499)	total: 4m 23s	remaining: 0us
bestTest = 0.4805097079
bestIteration = 499


CatBoostClassifier(cat_features=['content_type', 'age', 'sex', 'income'], depth=6, devices='0', early_stopping_rounds=50, eval_metric='Logloss', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, task_type='GPU', use_best_model=True, verbose=100)

In [17]:
del X_tr, X_val, y_tr, y_val
gc.collect()

print(f'Best iteration: {model.get_best_iteration()}')
print(f'Best Logloss: {model.get_best_score()["validation"]["Logloss"]:.4f}')

Best iteration: 499
Best Logloss: 0.4805


## Metrics

In [18]:
def compute_metrics(top_k_items, true_items, k=10):
    true_set = set(true_items)
    hits = 0
    sum_precision = 0.0
    sum_ndcg = 0.0
    for rank, item in enumerate(top_k_items[:k], start=1):
        if item in true_set:
            hits += 1
            sum_precision += hits / rank
            sum_ndcg += 1.0 / np.log2(rank + 1)
    num_relevant = min(k, len(true_set))
    if num_relevant == 0:
        return 0.0, 0.0, 0.0
    map_ = sum_precision / num_relevant
    ideal_dcg = sum(1.0 / np.log2(i + 1) for i in range(1, num_relevant + 1))
    ndcg = sum_ndcg / ideal_dcg if ideal_dcg > 0 else 0.0
    hr = 1.0 if hits > 0 else 0.0
    return hr, ndcg, map_

In [19]:
def evaluate(sampled_users, candidates, ground_truth, batch_size=20, k=10):
    hr_list, ndcg_list, map_list = [], [], []
    for i in tqdm(range(0, len(sampled_users), batch_size)):
        batch = sampled_users[i:i + batch_size]
        rows = [{'user_id': int(u), 'item_id': iid} for u in batch for iid in candidates[int(u)]]
        batch_df = pd.DataFrame(rows)
        batch_df = batch_df.merge(users_raw, on='user_id', how='left')
        batch_df = batch_df.merge(items_raw, on='item_id', how='left')
        for col in CAT_FEATURES:
            batch_df[col] = batch_df[col].astype(str)

        batch_df['score'] = model.predict_proba(batch_df[FEATURES])[:, 1]

        for user_id in batch:
            user_id = int(user_id)
            preds = batch_df[batch_df['user_id'] == user_id].sort_values('score', ascending=False)
            hr, ndcg, map_ = compute_metrics(preds['item_id'].head(k).tolist(), ground_truth.get(user_id, []), k=k)
            hr_list.append(hr)
            ndcg_list.append(ndcg)
            map_list.append(map_)
    return float(np.mean(hr_list)), float(np.mean(ndcg_list)), float(np.mean(map_list))

## Evaluate on train sample

In [20]:
hr_train, ndcg_train, map_train = evaluate(
    eval_data['train_sampled_users'],
    eval_data['train_candidates'],
    eval_data['train_ground_truth'],
)

100%|██████████| 25/25 [00:03<00:00,  7.04it/s]


## Evaluate on val sample

In [21]:
hr_val, ndcg_val, map_val = evaluate(
    eval_data['val_sampled_users'],
    eval_data['val_candidates'],
    eval_data['val_ground_truth'],
)

100%|██████████| 25/25 [00:03<00:00,  7.30it/s]


## Results

In [22]:
results = pd.DataFrame({
    'Metric': ['HR@10', 'NDCG@10', 'MAP@10'],
    'Train': [hr_train, ndcg_train, map_train],
    'Val': [hr_val, ndcg_val, map_val],
})
results['Gap'] = results['Train'] - results['Val']
print(results.to_string(index=False))

 Metric    Train      Val       Gap
  HR@10 0.294000 0.330000 -0.036000
NDCG@10 0.177701 0.163782  0.013920
 MAP@10 0.146750 0.126705  0.020045


## Save model

In [25]:
OUTPUT_DIR = '/kaggle/working/'
MODEL_PATH = f'{OUTPUT_DIR}catboost_raw_features.cbm'
model.save_model(MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')

Model saved to /kaggle/working/catboost_raw_features.cbm
